# Pipeline para ingesta de datos semiestructurados en batch - delta-delta-delta

Ingesta incremental de CSV desde MinIO hacia una arquitectura medallion (bronze → silver → gold) con tablas Delta.

> **Aviso antes de ejecutar.** Esta versión añade columnas nuevas a bronze (`archivo`, `ingest_time`) y
> cambia el esquema de la tabla de auditoría. Si ya tienes tablas creadas con la versión anterior del
> notebook, borra estas rutas en MinIO antes de empezar para evitar conflictos de esquema:
>
> - `hospital/bronze/spo2temperatura`
> - `hospital/bronze/spo2temperatura_checkpoint`
> - `hospital/silver/spo2temperatura`
> - `hospital/gold/spo2temperatura`
> - `hospital/audit/spo2temperatura`
>
> Los CSV de origen están en el bucket `spark`, no en `hospital`: asegúrate de que ambos existen.

## 1. Crea la sesión

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

JARS = ",".join([
    "/opt/spark/jars/hadoop-aws-3.4.0.jar",
    "/opt/spark/jars/aws-java-sdk-bundle-2.23.19.jar",
    "/opt/spark/jars/delta-spark_2.13-4.0.0.jar",
    "/opt/spark/jars/delta-storage-4.0.0.jar",
    "/opt/spark/jars/antlr4-runtime-4.13.1.jar",
    "/opt/spark/jars/mssql-jdbc-12.10.2.jre11.jar",
])

spark = (SparkSession.builder
    .appName("BatchSemiEstructurado")
    .master("spark://spark-master:7077")
    .config("spark.submit.deployMode", "client")

    # Limita los cores que toma esta aplicación. En standalone, por defecto una
    # aplicación se queda con todos los cores del cluster y las siguientes esperan
    # indefinidamente. Si vas a tener el notebook de streaming corriendo a la vez, bájalo a 1.
    .config("spark.cores.max", "2")

    # Extensiones Delta
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")

    # Crea el catálogo como un DeltaCatalog
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    .config("spark.sql.warehouse.dir", "s3a://hospital/")

    # Conexión a MinIO
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

    # Los jars ya están dentro de la imagen de los workers; aquí se cargan en el driver
    .config("spark.jars", JARS)

    .getOrCreate())

print("SparkSession creada")
print("Spark:", spark.version)

## 1.1 Evolución de esquema en Delta

Dos matices importantes sobre estas opciones:

- La clave de sesión para el *autoMerge* es `spark.databricks.delta.schema.autoMerge.enabled`.
  Mantiene el prefijo `databricks` también en la versión open source de Delta. Si se escribe
  `delta.schema.autoMerge.enabled` no da error: simplemente no hace nada.
- `delta.columnMapping.mode` **no** es una opción de sesión, es una propiedad de tabla. Se fija
  al crear la tabla con `.option("delta.columnMapping.mode", "name")` o con `TBLPROPERTIES`.
  Ponerla con `spark.conf.set` no tiene ningún efecto.

In [ ]:
# Mejora la planificación de consultas (opcional)
spark.conf.set("spark.sql.adaptive.enabled", "true")

# Permite que un MERGE o un append añadan columnas nuevas a la tabla destino
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# Comprobamos que la opción ha quedado realmente fijada
print("autoMerge:", spark.conf.get("spark.databricks.delta.schema.autoMerge.enabled"))

## 2. Define el esquema de ingestado

Recuerda: se está utilizando un catálogo en memoria, por lo que solo estará disponible en esta sesión.

In [ ]:
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, TimestampType)

schema = StructType([
    StructField("paciente_id", StringType(), True),
    StructField("temperatura", DoubleType(), True),
    StructField("SpO2", DoubleType(), True),
    StructField("timestamp", TimestampType(), True),
])

## 3. Crea la tabla de archivos procesados: checkpoint

Esta tabla es la que hace que la ingesta sea incremental: guarda qué archivos se han procesado ya,
para no volver a leerlos en la siguiente ejecución.

In [ ]:
path_processed_table = "s3a://hospital/bronze/spo2temperatura_checkpoint"

schema_processed = StructType([
    StructField("archivo", StringType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
])

if not DeltaTable.isDeltaTable(spark, path_processed_table):
    processed_files_df = spark.createDataFrame([], schema_processed)
    processed_files_df.write.format("delta").mode("overwrite").save(path_processed_table)
    print("Tabla de checkpoint creada (vacía)")
else:
    processed_files_df = spark.read.format("delta").load(path_processed_table)
    print(f"Tabla de checkpoint existente: {processed_files_df.count()} archivos registrados")

# Nota: collect() trae la lista al driver. Aceptable con decenas o centenares de
# archivos; con millones habría que resolverlo con un anti-join contra la tabla Delta.
processed_files = [row["archivo"] for row in processed_files_df.collect()]

## 4. Ingestar en bronze

Objetivo de bronze: guardar el dato **tal y como llega**, sin correcciones, para poder auditar y
reprocesar.

Dos decisiones de diseño:

- **Solo inserción (`append`), no MERGE.** Un `whenMatchedUpdateAll` sobre bronze sobrescribiría el
  dato original, que es justo lo que esta capa debe conservar. Además, un MERGE falla si dos filas
  del origen comparten la clave (`DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE`), y en
  estos CSV puede haber duplicados. La idempotencia la garantiza la tabla de checkpoint: un archivo
  ya procesado no se vuelve a leer.
- **Se añade la procedencia** (`archivo` e `ingest_time`), que es lo que permite auditar de dónde
  salió cada fila y reprocesar un archivo concreto.

In [ ]:
path_csvs = "s3a://spark/spo2temperatura/"   # ojo: bucket "spark", no "hospital"
bronze_path_table = "s3a://hospital/bronze/spo2temperatura"

# Listar archivos del directorio a través del FileSystem de Hadoop
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jvm.java.net.URI.create(path_csvs),
    spark._jsc.hadoopConfiguration()
)
status_list = fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path(path_csvs))

all_csv_files = [s.getPath().toString() for s in status_list
                 if s.getPath().toString().lower().endswith(".csv")]

new_files = [f for f in all_csv_files if f not in processed_files]

print(f"Archivos totales: {len(all_csv_files)} | nuevos: {len(new_files)}")
for f in new_files:
    print("  ", f)

if len(new_files) == 0:
    print("No hay archivos nuevos para procesar.")
    bronze_new_files = False
else:
    bronze_new_files = True

    df_bronze = (spark.read.format("csv")
        .option("header", "true")
        .schema(schema)
        .load(new_files)
        # Procedencia: de qué archivo viene cada fila y cuándo se ingestó
        .withColumn("archivo", F.input_file_name())
        .withColumn("ingest_time", F.current_timestamp()))

    # Bronze es append-only (ver nota de la celda anterior)
    (df_bronze.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .save(bronze_path_table))

    # Registrar los archivos procesados con las tres columnas del esquema
    nuevos = [(f, None, "OK") for f in new_files]
    new_files_df = (spark.createDataFrame(nuevos, schema_processed)
                    .withColumn("processed_at", F.current_timestamp()))
    new_files_df.write.format("delta").mode("append").save(path_processed_table)

    print(f"Ingestadas {df_bronze.count()} filas en bronze")

## 5. Ingesta en silver

Objetivo: datos fiables y tipados. Aquí sí corresponde deduplicar y descartar valores fuera de rango.

El MERGE usa la clave de negocio (`paciente_id`, `timestamp`), de modo que reejecutar la celda no
duplica filas. El `dropDuplicates` previo es imprescindible: sin él, dos filas del origen con la
misma clave hacen fallar el MERGE.

In [ ]:
silver_path_table = "s3a://hospital/silver/spo2temperatura"

if bronze_new_files:
    df_silver_new = (df_bronze
        .filter((F.col("temperatura") >= 20) & (F.col("temperatura") <= 50))
        .dropDuplicates(["paciente_id", "timestamp"]))

    if DeltaTable.isDeltaTable(spark, silver_path_table):
        delta_table = DeltaTable.forPath(spark, silver_path_table)
        (delta_table.alias("t").merge(
            df_silver_new.alias("s"),
            "t.paciente_id = s.paciente_id AND t.timestamp = s.timestamp")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print("Silver actualizada con MERGE")
    else:
        df_silver_new.write.format("delta").save(silver_path_table)
        print("Silver creada")

In [ ]:
if bronze_new_files:
    df_silver_new.show(truncate=False)
else:
    print("Sin datos nuevos en esta ejecución")

## 6. Ingestar en gold

Métricas por paciente, día y hora.

### Por qué no basta con filtrar por el timestamp mínimo

La versión anterior filtraba silver con `timestamp > min_timestamp` y agregaba el resultado. Eso
daba medias incorrectas, por dos motivos encadenados:

1. El `>` es estricto, así que excluía la propia fila con el timestamp mínimo.
2. Al corregirlo a `>=` aparece el problema de fondo: la media de un grupo
   (paciente, día, hora) se calculaba **solo con las filas nuevas**, dejando fuera las antiguas de
   ese mismo grupo. Y como el MERGE hace `whenMatchedUpdateAll`, la media correcta quedaba
   sobrescrita por una media parcial.

Ejemplo: el paciente 4 tiene lecturas a las 10:00 y 10:30, y gold guarda la media de ambas. Llega un
CSV con una lectura a las 10:45. Con `>` esa lectura se pierde; con `>=`, gold pasa a contener
únicamente la media de la lectura de las 10:45.

### Enfoque correcto

Seguir siendo incremental, pero recalculando **grupos completos**: primero se identifican los grupos
que toca el dato nuevo, y después se releen de silver todas las filas de esos grupos.

In [ ]:
gold_path_table = "s3a://hospital/gold/spo2temperatura"

if bronze_new_files:
    # 1. Grupos (paciente, fecha, hora) afectados por los datos nuevos
    claves = (df_silver_new
        .withColumn("fecha", F.to_date("timestamp"))
        .withColumn("hora", F.hour("timestamp"))
        .select("paciente_id", "fecha", "hora")
        .distinct())

    # 2. Todas las filas de silver, pero solo de esos grupos
    df_silver_afectado = (spark.read.format("delta").load(silver_path_table)
        .withColumn("fecha", F.to_date("timestamp"))
        .withColumn("hora", F.hour("timestamp"))
        .join(claves, ["paciente_id", "fecha", "hora"], "inner"))

    # 3. Recalcular la media completa de cada grupo afectado
    df_gold_new = (df_silver_afectado
        .groupBy("paciente_id", "fecha", "hora")
        .agg(F.avg("temperatura").alias("avg_temperatura"),
             F.avg("SpO2").alias("avg_SpO2")))

    if DeltaTable.isDeltaTable(spark, gold_path_table):
        delta_table = DeltaTable.forPath(spark, gold_path_table)
        (delta_table.alias("t").merge(
            df_gold_new.alias("s"),
            "t.paciente_id = s.paciente_id AND t.fecha = s.fecha AND t.hora = s.hora")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print("Gold actualizada con MERGE")
    else:
        df_gold_new.write.format("delta").save(gold_path_table)
        print("Gold creada")

In [ ]:
if bronze_new_files:
    df_gold_new.orderBy("paciente_id", "fecha", "hora").show(truncate=False)
else:
    print("Sin datos nuevos en esta ejecución")

## Tabla de estado actual del paciente (SCD1)

Las capas silver y gold anteriores responden a preguntas sobre el **pasado**: qué lecturas hubo y
cuál fue la media por hora. El proceso de alertas en streaming necesita otra cosa: saber **cómo está
cada paciente ahora**. Esa es la tabla que se construye aquí.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

silver_path = "s3a://hospital/silver/spo2temperatura"
scd1_path   = "s3a://hospital/gold/spo2temperatura_scd1"

assert DeltaTable.isDeltaTable(spark, silver_path), \
    "No existe silver: ejecuta primero el notebook de batch"

silver = spark.read.format("delta").load(silver_path)

# SCD1: nos quedamos con la última lectura de cada paciente
w = Window.partitionBy("paciente_id").orderBy(F.col("timestamp").desc())

scd1 = (silver
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn")
    .select("paciente_id",
            "temperatura",
            F.col("SpO2").alias("spo2"))
    # Atajo para la demo: los CSV son de diciembre de 2025 y la celda de alertas
    # solo acepta lecturas de las últimas 6 horas
    .withColumn("timestamp", F.current_timestamp()))

# El overwrite es lo que hace que sea SCD1: reemplaza el estado anterior
scd1.write.format("delta").mode("overwrite").save(scd1_path)

print(f"SCD1 creada: {scd1.count()} pacientes")
print("Pacientes que dispararán alerta si su ecg_avg supera 90:")
scd1.filter("temperatura < 30").orderBy("paciente_id").show()

## 7. Auditoría del proceso de ingesta

`run_at` es cuándo se ejecutó la ingesta; `min_ts_nuevos` es el tiempo de evento más antiguo de los
datos nuevos. En la versión anterior se guardaba solo lo segundo bajo el nombre `timestamp`, lo que
inducía a leerlo como la hora de la ejecución.

In [ ]:
audit_path_table = "s3a://hospital/audit/spo2temperatura"

if bronze_new_files:
    bronze_count = df_bronze.count()
    silver_count = df_silver_new.count()
    gold_count = df_gold_new.count()
    discarded = bronze_count - silver_count   # fuera de rango + duplicados
    min_ts_nuevos = df_silver_new.select(F.min("timestamp")).collect()[0][0]

    audit_df = (spark.createDataFrame(
            [(bronze_count, silver_count, gold_count, discarded,
              len(new_files), min_ts_nuevos)],
            "bronze_count long, silver_count long, gold_count long, "
            "discarded long, archivos long, min_ts_nuevos timestamp")
        .withColumn("run_at", F.current_timestamp()))

    audit_df.show(truncate=False)

    modo = "append" if DeltaTable.isDeltaTable(spark, audit_path_table) else "overwrite"
    audit_df.write.format("delta").mode(modo).option("mergeSchema", "true").save(audit_path_table)
    print(f"Auditoría guardada (modo {modo})")

## 8. Cerrar la sesión

Importante en el cluster standalone: mientras la aplicación siga viva retiene sus cores, y otra
sesión (otro notebook, un DAG de Airflow) se quedaría esperando sin ningún mensaje de error claro.

In [ ]:
spark.stop()